# 04 — Arm 3: Frozen retrieval shortlist + LLM selection

Local variant of `04_arm3_llm_colab.ipynb`. Identical logic; the Colab clone, input
upload and output-zip cells are removed because `data/` and `artefacts/` are already
present here. Arm 3 needs a GPU for the real run — on CPU, set `QUICK_SMOKE_TEST = True`,
which exercises prompt building, name matching, scoring and selection with a deterministic
stand-in generator and downloads nothing.

All substantive logic lives in `src/amlh/arm3_llm.py`. See the Colab notebook for the two
defects that invalidated the first Arm 3 run and the guards (§0.3, §2) that now catch them.


In [ ]:
QUICK_SMOKE_TEST = False

In [ ]:
import time
import zipfile
from pathlib import Path

import pandas as pd
import torch

from amlh import arm3_llm as a3
from amlh import features
from amlh.config import ARTEFACTS_DIR, FIGURES_DIR, HYPERPARAMETERS, PROJECT_ROOT, SEED, set_seed

### 0.3 Input integrity check

The check the first run did not have. `features.build_index` now raises on a missing
document, but failing here — before a model is downloaded — is cheaper and names the
cause directly.

In [ ]:
train = pd.read_csv(PROJECT_ROOT / "data" / "patient_qa_classification_train.csv")
assert {"question", "answer", "disease", "reference_url"} <= set(train.columns), train.columns
assert train["disease"].nunique() == 906, train["disease"].nunique()

# The "D" component of the frozen QLAD index variant. Missing documents are what
# silently turned QLAD into QLA on the first run.
coverage = features.doc_coverage(train["disease"].unique())
assert coverage["missing"] == [], f"{len(coverage['missing'])} classes have no NHS document: {coverage['missing'][:5]}"

for name in ("split_fit.csv", "split_val.csv", "arm1_val_predictions.csv"):
    assert (ARTEFACTS_DIR / name).is_file(), f"missing artefacts/{name} — rebuild arm3_colab_inputs.zip"

print(f"train: {len(train)} rows, {train['disease'].nunique()} classes")
print(f"NHS documents: {coverage['n_found']}/{coverage['n_total']} classes covered")
print("artefacts present: split_fit.csv, split_val.csv, arm1_val_predictions.csv")

## 1. Load splits from `artefacts/`

In [ ]:
set_seed()

fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")
arm1_predictions = pd.read_csv(ARTEFACTS_DIR / "arm1_val_predictions.csv")

if QUICK_SMOKE_TEST:
    keep = val["disease"].head(10).tolist()
    val = val[val["disease"].isin(keep)].head(10).reset_index(drop=True)
    arm1_predictions = arm1_predictions.head(len(val)).reset_index(drop=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"fit={len(fit)} ({fit.disease.nunique()} classes) | val={len(val)} ({val.disease.nunique()} classes)")
print(f"device={device} | gpu={gpu_name}")

shortlist_k = a3.shortlist_k(HYPERPARAMETERS)
n_shots = a3.n_shots(HYPERPARAMETERS)
primary_model = a3.selected_model_name(HYPERPARAMETERS)
secondary_model = a3.secondary_model_name(HYPERPARAMETERS)
prompt_modes = ["zero_shot", "few_shot", "cot"]

print({
    "shortlist_k": shortlist_k,
    "llm_temperature": a3.llm_temperature(HYPERPARAMETERS),
    "n_shots": n_shots,
    "max_new_tokens": a3.max_new_tokens_for("zero_shot"),
    "cot_max_new_tokens": a3.max_new_tokens_for("cot"),
    "primary_model": primary_model,
    "secondary_model": secondary_model,
    "prompt_mode": HYPERPARAMETERS.prompt_mode,  # None until this notebook selects it
})

## 2. Frozen Arm 1 shortlist — and proof it is the frozen one

`build_shortlist_ranking` runs the frozen Arm 1 configuration from `config.py`. The
assertion below then checks rank 1 against `artefacts/arm1_val_predictions.csv`, item for
item. On the first run this would have failed on 17/200 items and stopped the notebook
before a single prompt was sent.

In [ ]:
set_seed()
shortlist_rankings, top_sim = a3.build_shortlist_ranking(fit, val, HYPERPARAMETERS, depth=shortlist_k)

check = a3.assert_reproduces_arm1(shortlist_rankings, arm1_predictions)
print(f"shortlist top-1 reproduces Arm 1 on all {check['n']} items")

arm1_top1 = [ranking[0] for ranking in shortlist_rankings]
gold = val["disease"].tolist()
in_shortlist = sum(g in ranking for g, ranking in zip(gold, shortlist_rankings)) / len(gold)
print(f"Arm 1 top-1 accuracy on these items : {sum(p == g for p, g in zip(arm1_top1, gold)) / len(gold):.3f}")
print(f"gold label present in the shortlist  : {in_shortlist:.3f}  (the ceiling the LLM selects against)")

## 3. Prompt budget check

Prompt lengths are measured with the primary model's tokeniser before any condition runs,
and the truncation rate is printed explicitly.

In [ ]:
from transformers import AutoTokenizer

set_seed()
examples = a3.build_examples(fit, n=n_shots, seed=SEED)
budget_tokeniser = AutoTokenizer.from_pretrained(primary_model)

prompts_by_mode = {}
budget_rows = []
for mode in prompt_modes:
    prompts = a3.build_prompts_for_condition(
        val, shortlist_rankings, mode, examples=examples if mode == "few_shot" else None
    )
    prompts_by_mode[mode] = prompts
    summary = a3.prompt_token_lengths(prompts, budget_tokeniser, max_length=512)
    budget_rows.append({
        "condition": mode,
        "min_tokens": summary["min"],
        "median_tokens": summary["median"],
        "mean_tokens": summary["mean"],
        "p95_tokens": summary["p95"],
        "max_tokens": summary["max"],
        "truncation_rate_512": summary["truncation_rate"],
    })

budget_df = pd.DataFrame(budget_rows)
print(budget_df.to_string(index=False))
print()
print("--- one zero-shot prompt, as the model receives it ---")
print(a3.flatten_messages(prompts_by_mode["zero_shot"][0]))

## 4. The generator

`a3.load_generator` returns `(tokenizer, model, pipe)`, where `pipe` carries the
practical's signature and `[{"generated_text": ...}]` return shape. Causal checkpoints
(MediPhi) go through `apply_chat_template`; encoder-decoder checkpoints (Flan-T5) have no
chat template, so their messages are flattened to text.

In [ ]:
def smoke_pipe(prompt, max_new_tokens=100, temperature=0.0, **kwargs):
    # Deterministic stand-in that answers with the first candidate's name, so the
    # smoke run exercises prompt building, name matching, scoring and selection
    # end to end without downloading a model.
    user = prompt[-1]["content"]
    first_candidate = user.split("Diagnoses:\n")[1].split("\n")[0].split(", ")[0]
    return [{"generated_text": f"Final answer: {first_candidate}"}]


set_seed()
if QUICK_SMOKE_TEST:
    primary_pipe = smoke_pipe
else:
    _, primary_model_obj, primary_pipe = a3.load_generator(primary_model, device=device)
    print(f"loaded {primary_model} on {device}")

## 5. Run the three prompt conditions on the primary model

In [ ]:
set_seed()
condition_frames = {}
condition_rows = []

for mode in prompt_modes:
    start = time.perf_counter()
    pred_df, metrics, _ = a3.run_condition(
        fit,
        val,
        HYPERPARAMETERS,
        mode=mode,
        pipe=primary_pipe,
        examples=examples if mode == "few_shot" else None,
        shortlist_depth=shortlist_k,
        model_name=primary_model,
        shortlist_rankings=shortlist_rankings,
        top_sim=top_sim,
    )
    metrics["wall_clock_sec"] = time.perf_counter() - start
    condition_frames[mode] = pred_df
    condition_rows.append(metrics)
    print(f"{mode:>10}: acc={metrics['accuracy']:.3f}  arm1={metrics['arm1_accuracy']:.3f}  "
          f"fallback={metrics['fallback_rate']:.3f}  ({metrics['wall_clock_sec']:.0f}s)")

condition_df = pd.DataFrame(condition_rows)
print()
print(condition_df.to_string(index=False))

### 5.1 Did the LLM earn its place?

`accuracy_minus_arm1` is the quantity the first run never recorded: the LLM re-ranks a
shortlist whose top-1 is already a prediction, so the retriever is the baseline it has to
beat. McNemar pairs each condition against that top-1 over the same items.

In [ ]:
vs_arm1_df = a3.condition_vs_arm1_mcnemar(condition_frames)
print(vs_arm1_df.to_string(index=False))

## 6. Prompt-condition selection

Pairwise McNemar over the same validation items. The rule is pre-registered in CLAUDE.md:
if no condition separates from the others at p < 0.05 the comparison is **reported as
unresolved** and `zero_shot` is kept on the declared prior of the simplest prompt. That
can retain a lower-scoring condition, which is the intended behaviour of a prior.

In [ ]:
mcnemar_df = a3.pairwise_condition_mcnemar(condition_frames)
selected_mode, condition_tie_break_fired = a3.select_prompt_mode(condition_df, mcnemar_df)

print(mcnemar_df.to_string(index=False))
print()
print({"selected_mode": selected_mode, "tie_break_fired": condition_tie_break_fired})
if condition_tie_break_fired:
    print("UNRESOLVED: no condition separates at p < 0.05; zero_shot kept on the declared prior.")

## 7. Model ablation

The same three conditions on the secondary generator. Selection order is pre-registered:
the prompt condition was chosen on the primary model alone (§6), and the model is chosen
at that condition (§8). The other rows are reported for transparency and never select
anything — a 6-cell grid on a 200-item hold-out (SE ≈ 3.5pp) would manufacture a choice
the data cannot support.

In [ ]:
if not QUICK_SMOKE_TEST:
    del primary_model_obj
    torch.cuda.empty_cache()

set_seed()
if QUICK_SMOKE_TEST:
    secondary_pipe = smoke_pipe
else:
    _, secondary_model_obj, secondary_pipe = a3.load_generator(secondary_model, device=device)
    print(f"loaded {secondary_model} on {device}")

secondary_frames = {}
secondary_rows = []
for mode in prompt_modes:
    start = time.perf_counter()
    pred_df, metrics, _ = a3.run_condition(
        fit,
        val,
        HYPERPARAMETERS,
        mode=mode,
        pipe=secondary_pipe,
        examples=examples if mode == "few_shot" else None,
        shortlist_depth=shortlist_k,
        model_name=secondary_model,
        shortlist_rankings=shortlist_rankings,
        top_sim=top_sim,
    )
    metrics["wall_clock_sec"] = time.perf_counter() - start
    secondary_frames[mode] = pred_df
    secondary_rows.append(metrics)
    print(f"{mode:>10}: acc={metrics['accuracy']:.3f}  arm1={metrics['arm1_accuracy']:.3f}  "
          f"fallback={metrics['fallback_rate']:.3f}  ({metrics['wall_clock_sec']:.0f}s)")

ablation_df = pd.concat([condition_df, pd.DataFrame(secondary_rows)], ignore_index=True)
print()
print(ablation_df.to_string(index=False))

## 8. Model selection

McNemar at the selected condition, mirroring the Arm 2 encoder rule: if the comparison is
unresolved at p ≥ 0.05 it is **reported as unresolved** and the in-domain clinical model is
kept on the declared prior. This too can retain the lower-scoring model.

In [ ]:
model_mcnemar_df = a3.model_mcnemar(condition_frames[selected_mode], secondary_frames[selected_mode])
selected_model, model_tie_break_fired = a3.select_model(model_mcnemar_df, HYPERPARAMETERS)

print(f"comparison made at the selected condition: {selected_mode}")
print(model_mcnemar_df.to_string(index=False))
print()
print({"selected_model": selected_model, "tie_break_fired": model_tie_break_fired})
if model_tie_break_fired:
    print(f"UNRESOLVED: McNemar does not separate the generators; {selected_model} kept on the declared prior.")

## 9. Persist artefacts

Nothing below reads a test-set quantity. The values printed at the end are the ones to copy
into `config.py`.

In [ ]:
predictions_df = pd.concat(
    list(condition_frames.values()) + list(secondary_frames.values()), ignore_index=True
)
predictions_df.to_csv(ARTEFACTS_DIR / "arm3_val_predictions.csv", index=False)
ablation_df.to_csv(ARTEFACTS_DIR / "arm3_prompt_conditions.csv", index=False)
mcnemar_df.to_csv(ARTEFACTS_DIR / "arm3_condition_mcnemar.csv", index=False)
vs_arm1_df.to_csv(ARTEFACTS_DIR / "arm3_vs_arm1_mcnemar.csv", index=False)
model_mcnemar_df.to_csv(ARTEFACTS_DIR / "arm3_model_mcnemar.csv", index=False)
budget_df.to_csv(ARTEFACTS_DIR / "arm3_prompt_budget.csv", index=False)
with open(ARTEFACTS_DIR / "arm3_prompts.txt", "w", encoding="utf-8") as f:
    f.write(a3.build_prompt_table(prompts_by_mode))

print("--- copy into config.py ---")
print({
    "shortlist_k": shortlist_k,
    "llm_temperature": a3.llm_temperature(HYPERPARAMETERS),
    "n_shots": n_shots,
    "prompt_mode": selected_mode,
    "arm3_model_name": selected_model,
})
print()
print(f"prompt-condition tie-break fired: {condition_tie_break_fired}")
print(f"model tie-break fired           : {model_tie_break_fired}")